In [3]:
import numpy as np
import pandas as pd

CAPACITY_REDUCTION_FACTOR = 0.65
STEEL_MODULUS_MPA = 200000
ULTIMATE_CONCRETE_STRAIN = 0.003
RECIPROCAL_METHOD_MIN_AXIAL_RATIO = 0.1

SECTION_DIM_X_MM = 914.4
SECTION_DIM_Y_MM = 609.6
FC_PRIME_MPA = 30
FY_MPA = 420
COVER_TO_BAR_CENTROID_MM = 60
BAR_AREA_MM2 = 819
BARS_ALONG_X_FACE = 8
BARS_ALONG_Y_FACE = 5

# Upper Pu sits just above phi times P0 so that the whole safe range is covered
PU_VALUES_KN = np.arange(1000, 14001, 250)


def build_eccentricity_grid(half_dimension_mm):
    # 10 mm steps over the first half of the quadrant, 5 mm steps over the second half
    coarse_steps = np.arange(0, half_dimension_mm / 2, 10)
    fine_steps = np.arange(coarse_steps[-1] + 5, half_dimension_mm, 5)
    return np.concatenate([coarse_steps, fine_steps])


def solve_capacity_at_eccentricity(depth, cover, intermediate_bars, end_face_intermediate_bars, bar_area, fc, fy, beta_1, width, eccentricity):
    # Strain compatibility with the Whitney block, neutral axis found by bisection on M - e * P
    layer_index = np.arange(intermediate_bars.max())[None, :]
    layer_active = layer_index < intermediate_bars[:, None]
    middle_depths = cover[:, None] + (depth - 2 * cover)[:, None] * (layer_index + 1) / (intermediate_bars + 1)[:, None]
    layer_depths = np.concatenate([cover[:, None], middle_depths, (depth - cover)[:, None]], axis = 1)
    end_count = (2 + end_face_intermediate_bars)[:, None]
    layer_counts = np.concatenate([end_count, 2 * layer_active, end_count], axis = 1)

    def force_and_moment(neutral_axis):
        block_depth = np.minimum(beta_1 * neutral_axis, depth)
        concrete_force = 0.85 * fc * block_depth * width
        strain = ULTIMATE_CONCRETE_STRAIN * (neutral_axis[:, None] - layer_depths) / neutral_axis[:, None]
        steel_stress = np.clip(STEEL_MODULUS_MPA * strain, -fy[:, None], fy[:, None])
        displaced_concrete = 0.85 * fc[:, None] * (layer_depths < block_depth[:, None])
        steel_force = layer_counts * bar_area[:, None] * (steel_stress - displaced_concrete)
        axial_force = concrete_force + steel_force.sum(axis = 1)
        moment = concrete_force * (depth - block_depth) / 2 + (steel_force * (depth[:, None] / 2 - layer_depths)).sum(axis = 1)
        return axial_force, moment

    lower_bound = np.full(depth.shape, 1.0)
    upper_bound = 20 * depth
    for _ in range(70):
        midpoint = (lower_bound + upper_bound) / 2
        axial_force, moment = force_and_moment(midpoint)
        too_much_moment = (moment - eccentricity * axial_force) > 0
        lower_bound = np.where(too_much_moment, midpoint, lower_bound)
        upper_bound = np.where(too_much_moment, upper_bound, midpoint)
    neutral_axis = (lower_bound + upper_bound) / 2
    axial_force, _ = force_and_moment(neutral_axis)
    return axial_force / 1000, neutral_axis


# Section properties
number_of_bars = 2 * BARS_ALONG_X_FACE + 2 * (BARS_ALONG_Y_FACE - 2)
gross_area_mm2 = SECTION_DIM_X_MM * SECTION_DIM_Y_MM
steel_area_mm2 = number_of_bars * BAR_AREA_MM2
steel_ratio = steel_area_mm2 / gross_area_mm2
bar_diameter_mm = np.sqrt(4 * BAR_AREA_MM2 / np.pi)
beta_1 = float(np.clip(0.85 - 0.05 * (FC_PRIME_MPA - 28) / 7, 0.65, 0.85))
axial_capacity_p0_kn = (0.85 * FC_PRIME_MPA * (gross_area_mm2 - steel_area_mm2) + FY_MPA * steel_area_mm2) / 1000

# One quadrant is enough because the section and bar layout are symmetric about both axes
ex_values_mm = build_eccentricity_grid(SECTION_DIM_X_MM / 2)
ey_values_mm = build_eccentricity_grid(SECTION_DIM_Y_MM / 2)


def capacity_for_eccentricities(eccentricities, depth_mm, width_mm, intermediate_bars, end_face_intermediate_bars):
    count = len(eccentricities)
    capacity_kn, neutral_axis_mm = solve_capacity_at_eccentricity(
        depth = np.full(count, depth_mm), cover = np.full(count, COVER_TO_BAR_CENTROID_MM),
        intermediate_bars = np.full(count, intermediate_bars), end_face_intermediate_bars = np.full(count, end_face_intermediate_bars),
        bar_area = np.full(count, BAR_AREA_MM2), fc = np.full(count, FC_PRIME_MPA), fy = np.full(count, FY_MPA),
        beta_1 = np.full(count, beta_1), width = np.full(count, width_mm), eccentricity = eccentricities)
    # Zero eccentricity is pure axial load, so the capacity is P0 by definition
    capacity_kn = np.where(eccentricities == 0, axial_capacity_p0_kn, capacity_kn)
    return capacity_kn, neutral_axis_mm


# ex acts along x (depth 914.4 mm, bending about the y axis), ey acts along y (depth 609.6 mm, bending about the x axis)
pny_values_kn, neutral_axis_y_values_mm = capacity_for_eccentricities(ex_values_mm, SECTION_DIM_X_MM, SECTION_DIM_Y_MM, BARS_ALONG_X_FACE - 2, BARS_ALONG_Y_FACE - 2)
pnx_values_kn, neutral_axis_x_values_mm = capacity_for_eccentricities(ey_values_mm, SECTION_DIM_Y_MM, SECTION_DIM_X_MM, BARS_ALONG_Y_FACE - 2, BARS_ALONG_X_FACE - 2)

# Capacity depends only on (ex, ey), so build that table first and then cross it with Pu
ex_grid, ey_grid = np.meshgrid(np.arange(len(ex_values_mm)), np.arange(len(ey_values_mm)), indexing = "ij")
ex_index = ex_grid.ravel()
ey_index = ey_grid.ravel()
capacity_table = pd.DataFrame({
    "ex_mm": ex_values_mm[ex_index],
    "ey_mm": ey_values_mm[ey_index],
    "axial_capacity_Pnx_kN": pnx_values_kn[ey_index],
    "neutral_axis_x_mm": neutral_axis_x_values_mm[ey_index],
    "axial_capacity_Pny_kN": pny_values_kn[ex_index],
    "neutral_axis_y_mm": neutral_axis_y_values_mm[ex_index],
})
capacity_table["axial_capacity_P0_kN"] = axial_capacity_p0_kn
capacity_table["biaxial_capacity_Pni_kN"] = 1 / (1 / capacity_table["axial_capacity_Pnx_kN"] + 1 / capacity_table["axial_capacity_Pny_kN"] - 1 / capacity_table["axial_capacity_P0_kN"])
capacity_table["phi"] = CAPACITY_REDUCTION_FACTOR
capacity_table["phi_Pni_kN"] = capacity_table["phi"] * capacity_table["biaxial_capacity_Pni_kN"]
capacity_table["reciprocal_method_valid"] = capacity_table["biaxial_capacity_Pni_kN"] >= RECIPROCAL_METHOD_MIN_AXIAL_RATIO * FC_PRIME_MPA * gross_area_mm2 / 1000

extended_dataset = capacity_table.loc[capacity_table.index.repeat(len(PU_VALUES_KN))].reset_index(drop = True)
extended_dataset["pu_kN"] = np.tile(PU_VALUES_KN, len(capacity_table))
extended_dataset["utilization_ratio"] = extended_dataset["pu_kN"] / extended_dataset["phi_Pni_kN"]
extended_dataset["status"] = np.where(extended_dataset["phi_Pni_kN"] >= extended_dataset["pu_kN"], "SAFE", "UNSAFE")

section_columns = {
    "column_dim_x_mm": SECTION_DIM_X_MM, "column_dim_y_mm": SECTION_DIM_Y_MM, "fc_prime_MPa": FC_PRIME_MPA, "fy_MPa": FY_MPA,
    "cover_to_bar_centroid_mm": COVER_TO_BAR_CENTROID_MM, "number_of_bars": number_of_bars, "bar_number": "#32",
    "bar_area_mm2": BAR_AREA_MM2, "bar_diameter_mm": bar_diameter_mm, "gross_area_mm2": gross_area_mm2,
    "steel_area_mm2": steel_area_mm2, "steel_ratio": steel_ratio, "beta_1": beta_1,
}
for column_name, column_value in section_columns.items():
    extended_dataset[column_name] = column_value

extended_column_order = ["pu_kN", "ex_mm", "ey_mm"] + list(section_columns.keys()) + [
    "axial_capacity_P0_kN", "axial_capacity_Pnx_kN", "neutral_axis_x_mm", "axial_capacity_Pny_kN", "neutral_axis_y_mm",
    "biaxial_capacity_Pni_kN", "reciprocal_method_valid", "phi", "phi_Pni_kN", "utilization_ratio", "status",
]
extended_dataset = extended_dataset[extended_column_order].round(4)
extended_dataset.insert(0, "sr_no", np.arange(1, len(extended_dataset) + 1))

project_dataset = extended_dataset[["pu_kN", "ex_mm", "ey_mm", "status"]]

display(project_dataset.head())
display(project_dataset.tail())
print(project_dataset["status"].value_counts())

,pu_kN,ex_mm,ey_mm,status
0,1000,0.0,0.0,SAFE
1,1250,0.0,0.0,SAFE
2,1500,0.0,0.0,SAFE
3,1750,0.0,0.0,SAFE
4,2000,0.0,0.0,SAFE


,pu_kN,ex_mm,ey_mm,status
170655,13000,455.0,300.0,UNSAFE
170656,13250,455.0,300.0,UNSAFE
170657,13500,455.0,300.0,UNSAFE
170658,13750,455.0,300.0,UNSAFE
170659,14000,455.0,300.0,UNSAFE


status
UNSAFE    116901
SAFE       53759
Name: count, dtype: int64


In [4]:
project_dataset.to_csv("25280058_AI631_Project_Dataset.csv", index = False)

In [5]:
extended_dataset.to_csv("25280058_AI631_Project_Dataset_Extended.csv", index = False)